In [1]:
# 🧬 Requisitos
!pip install biopython

from Bio import AlignIO
import pandas as pd

# ===== Configura aquí =====
msa_path = "/content/alineamientoamya.fasta"  # ruta a tu MSA
columna_1base = 2179                          # columna "como en Jalview" (empieza en 1)
residuo_objetivo = "S"                        # un AA (p.ej. 'D', 'G', '-')
case_insensitive = True                       # True = ignora mayúsculas/minúsculas
contar_gap_como_match = True                  # True = si residuo_objetivo=='-' sí hace match con gaps
exportar_csv = True                           # Guardar resultados
csv_path = f"/content/IDs_col{columna_1base}_{residuo_objetivo}.csv"
# ==========================

def ids_con_residuo_en_columna(msa_path, columna_1base, residuo_objetivo,
                               case_insensitive=True, contar_gap_como_match=True):
    # Validaciones de entrada
    if residuo_objetivo is None or str(residuo_objetivo).strip() == "":
        raise ValueError("Debes proporcionar un residuo objetivo no vacío (e.g., 'D', 'A', '-').")

    # Normalización del residuo objetivo
    objetivo = str(residuo_objetivo).strip()
    if case_insensitive:
        objetivo = objetivo.upper()

    # Leer alineamiento
    alignment = AlignIO.read(msa_path, "fasta")

    # Validar rango de columna
    L = alignment.get_alignment_length()
    if not (1 <= columna_1base <= L):
        raise IndexError(f"La columna {columna_1base} está fuera de rango (1..{L}).")

    col_idx = columna_1base - 1  # a índice base-0 para Python

    ids_match = []
    for record in alignment:
        # Obtener residuo en la columna
        aa = str(record.seq[col_idx])
        # Normalizar según preferencia
        if case_insensitive:
            aa = aa.upper()

        # Lógica de match
        if objetivo == "-":
            # Quiero hacer match con gaps
            if contar_gap_como_match and aa == "-":
                ids_match.append(record.id)
        else:
            # Match exacto con el residuo objetivo (no gap)
            if aa == objetivo:
                ids_match.append(record.id)

    return ids_match, L, len(alignment)

# Ejecutar búsqueda
ids, L, N = ids_con_residuo_en_columna(msa_path, columna_1base, residuo_objetivo,
                                       case_insensitive=case_insensitive,
                                       contar_gap_como_match=contar_gap_como_match)

print(f"🔎 MSA: {msa_path}")
print(f"🧱 Largo del alineamiento (columnas): {L}")
print(f"👥 Número de secuencias: {N}")
print(f"📍 Columna consultada (base-1): {columna_1base}")
print(f"🎯 Residuo objetivo: {residuo_objetivo!r}")
print(f"✅ Secuencias que coinciden: {len(ids)}")

# Mostrar algunas (si son muchas)
for i, sid in enumerate(ids[:20], 1):
    print(f"{i:>3}. {sid}")
if len(ids) > 20:
    print(f"... y {len(ids)-20} más.")

# Exportar si se desea
if exportar_csv:
    df_out = pd.DataFrame({"sequence_id": ids})
    df_out.to_csv(csv_path, index=False)
    print(f"💾 IDs exportados a: {csv_path}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.2 MB/s eta 0:00:00
🔎 MSA: /content/alineamientoamya.fasta
🧱 Largo del alineamiento (columnas): 4550
👥 Número de secuencias: 362
📍 Columna consultada (base-1): 2179
🎯 Residuo objetivo: 'S'
✅ Secuencias que coinciden: 35
  1. Hb22
  2. Tb7
  3. Tb8
  4. Tb10
  5. Tb12
  6. Hb144
  7. Tb60
  8. Tb67
  9. Tb75
 10. Tb76
 11. Tb78
 12. Hb85
 13. Hb121
 14. Te34
 15. Te36
 16. Te42
 17. Te57
 18. Te6
 19. Te9
 20. Te15
... y 15 más.
💾 IDs exportados a: /content/IDs_col2179_S.csv
